# Build Forward propagation from scratch

### Basics - Understanding through a basic NN structure
This section focuses on building forward pass for the below neuron image.
<img src="http://cocl.us/neural_network_example" alt="Neural Network Example" width="600px">

Focuses on,
1. Compute weighted sum at each node
2. Compute Node Activation using Sigmoid function $$\sigma(x) = \frac{1}{1 + e^{-x}}$$
3. Use Forward Propagation to propagate the data


In [8]:
import numpy as np

weights = np.around(np.random.uniform(size = 6),decimals=2) # assign 6 weights random
biases = np.around(np.random.uniform(size = 3),decimals=2) # assign 3 random biases
print(weights)
print(biases)

# assign arbitrary values to the inputs x1 and x2
x1 = 0.5
x2 = 0.85

# compute weighted sum of nodes in hidden layer
z11 = weights[0] * x1 + weights[1] * x2 + biases[0]
z12 = weights[2] * x1 + weights[3] * x2 + biases[1]

print('The weighted sum of the inputs at first layer - 1st node:{}'.format(np.around(z11,decimals=2)))
print('The weighted sum of the inputs at first layer - 2nd node:{}'.format(np.around(z12,decimals=2)))

# Activation values a11 and a12 for the nodes - using Sigmoid function
a11 = 1/(1 + np.exp(-z11))
a12 = 1/(1 + np.exp(-z12))

print('Activation Values')
print('A11: {}'.format(np.around(a11,decimals=2)))
print('A12: {}'.format(np.around(a12,decimals=2)))

# Compute weighted sum at the output layer
z2 = weights[4] * a11 + weights[5] * a12 + biases[2]

# Compute a2 - sigmoid function
a2 = 1/(1 + np.exp(-z2))

# The prediction made by the network by taking the inputs x1 and x2
print('The output of the network for x1 = 0.5 and x2 = 0.85 is {}'.format(np.around(a2, decimals=4)))


[0.53 0.72 0.23 0.36 0.58 0.18]
[0.21 1.   0.28]
The weighted sum of the inputs at first layer - 1st node:1.09
The weighted sum of the inputs at first layer - 2nd node:1.42
Activation Values
A11: 0.75
A12: 0.81
The output of the network for x1 = 0.5 and x2 = 0.85 is 0.7024


### Build Neural Network
Now the focus is to compute the Neural Network for a real time case where we have many hidden layers and many more nodes in each layer.

Let's generalize the network in order to code the automatic way of predictions. Let's code this way to make the NN to take $N$ inputs, would have many hidden layers with each of them having $m$ nodes and an output layer and build the neural network

In [18]:
  # Initialize weights and biases for each layer
  # Steps: Loop through each layers including the output layer
  # Assgin Layer name, number of nodes.
  # Loop through each nodes and assign weights and bias of each node

# Start Building the NN - First step - Initialize weights and bias of each node
def initialize_network(num_inputs, num_hidden_layers, m_nodes_hidden, num_nodes_output):
    # Initialize weights and biases for each layer
    network = {}
    num_nodes_previous = num_inputs

    for layer in range(num_hidden_layers + 1):
        if layer == num_hidden_layers:
            layer_name = 'output'
            num_nodes = num_nodes_output
        else:
            layer_name = 'layer_{}'.format(layer + 1)
            num_nodes = m_nodes_hidden[layer]

        network[layer_name] = {}
        for node in range(num_nodes):
            node_name = 'node_{}'.format(node + 1)
            network[layer_name][node_name] = {
                'weights': np.around(np.random.uniform(size=num_nodes_previous), decimals=2),
                'bias': np.around(np.random.uniform(size=1), decimals=2)
            }
        num_nodes_previous = num_nodes

    return network


In [19]:
#  Call the Initialize network with 5 inputs, 3 hidden layers, 3 nodes in 1st, 2 nodes in 2nd and 3 nodes in 3rd layers, 1 node in output layer
n = 5 # number of inputs
num_hidden_layers = 3
m = [3,2,3] # number of nodes in each hidden layer
num_nodes_output = 1
print(initialize_network(n,num_hidden_layers,m,num_nodes_output))

{'layer_1': {'node_1': {'weights': array([0.21, 0.99, 0.6 , 0.34, 0.68]), 'bias': array([0.43])}, 'node_2': {'weights': array([0.07, 0.2 , 0.85, 0.8 , 0.51]), 'bias': array([1.])}, 'node_3': {'weights': array([0.93, 0.36, 0.16, 0.35, 0.77]), 'bias': array([0.01])}}, 'layer_2': {'node_1': {'weights': array([0.49, 0.98, 0.29]), 'bias': array([0.08])}, 'node_2': {'weights': array([0.72, 0.16, 0.63]), 'bias': array([0.57])}}, 'layer_3': {'node_1': {'weights': array([0.57, 0.2 ]), 'bias': array([0.72])}, 'node_2': {'weights': array([0.84, 0.78]), 'bias': array([0.94])}, 'node_3': {'weights': array([0.6 , 0.17]), 'bias': array([0.62])}}, 'output': {'node_1': {'weights': array([0.03, 0.91, 0.67]), 'bias': array([0.87])}}}


In [22]:
# Compute Weighted sum
def compute_weighted_sum(inputs,weights,bias):
  return np.sum(inputs * weights) + bias
# compute Activation function
def compute_activation(weighted_sum):
  return 1.0/(1.0 + np.exp(-1 * weighted_sum))

#Initialize random inputs to feed to the network
np.random.seed(123)
inputs = np.around(np.random.uniform(5),decimals=2)



### Compute Forward Propagation
 We are now putting all pieces together. Initialize the network, compute weighted sum of each node and compute its activation function to feed forward to the neural network. Finally, predict the output layer prediction by propagating all the way to there.

  **Approach:**
1. Start with the input layer as the input to the first hidden layer.
2. Compute the weighted sum at the nodes of the current layer.
3. Compute the output of the nodes of the current layer.
4. Set the output of the current layer to be the input to the next layer.
5. Move to the next layer in the network.
6. Repeat steps 2 - 5 until we compute the output of the output layer.

In [26]:
def forward_propagate(network,inputs):
  layer_inputs = list(inputs) # start with input layer
  for layer in network:
    layer_data = network[layer]
    layer_outputs = []

    for layer_node in layer_data:
      node_data = layer_data[layer_node]
      # compute weighted sum and activation as output to each node
      weighted_sum = compute_weighted_sum(layer_inputs,node_data['weights'],node_data['bias'])
      node_output = compute_activation(weighted_sum)

      layer_outputs.append(np.around(node_output,2))
    if layer != 'output':
      print('The outputs of the nodes in hidden layer number {} is {}'.format(layer.split('_')[1], layer_outputs))

    layer_inputs = layer_outputs # set the activation ouputs as inputs to the next node

  network_predictions = layer_outputs #output the final predicted values
  return network_predictions


### Putting the pieces together

In [29]:
#Step 1: Initialize the network
network = initialize_network(5,3,[2,3,2],2)
# Inputs
inputs = np.around(np.random.uniform(size=5), decimals=2)
# Compute the network Predictions
predictions = forward_propagate(network, inputs)
print('The predicted values by the network for the given input are {}'.format(predictions))


The outputs of the nodes in hidden layer number 1 is [array([0.96]), array([0.88])]
The outputs of the nodes in hidden layer number 2 is [array([0.87]), array([0.91]), array([0.89])]
The outputs of the nodes in hidden layer number 3 is [array([1.]), array([0.97])]
The predicted values by the network for the given input are [array([0.88]), array([0.8])]


*italicized text*# New Section